# OIM3600 — PEMS Homework: Investigate, Validate, and Transform Market Data

**Goal:** Write your own Python code to investigate three external data sources, test their quality and consistency, and build the `market_data` dictionary used by PEMS.

**What you are given:** The external JSON files and the basic data-loading code. **What you must write:** All eight validation checks and the code to construct `market_data`. No completed validation function is provided.

**Submission:** Save this notebook with your completed Python code cells and their executed output. **All student answers must be in code cells.** Use `print()` statements to report findings and Python comments to explain reasoning. Do not add written answers in Markdown cells or paste a prewritten validation function; develop each check yourself.

## Preparation — Load the external data

Place the notebook where the data directory can be found. The cell below searches for a `data` folder in the notebook's current working directory or its parent. Run it before starting the investigation. The existing `load_data()` function in PEMS contains additional PM and menu handling; this assignment focuses on the three market-data sources.

In [5]:
import json
from pathlib import Path
from pprint import pprint as pp

DATA = Path.cwd().parent.parent / "10-26M15/Classwork/data"
print(DATA)

with open(DATA / "nyse_top100.json", encoding="utf-8") as file:
    tickers = json.load(file)["tickers"]
with open(DATA / "corp_acts_yf.json", encoding="utf-8") as file:
    corporate_actions = json.load(file)["data"]
with open(DATA / "raw_eod_yf.json", encoding="utf-8") as file:
    eod_prices = json.load(file)["data"]

print("External sources loaded.")

c:\PythonClass\class_repo\classes\10-26M15\Classwork\data
External sources loaded.


## Part A — External data investigation and validation

Write code in **each section**. Output enough evidence to support your conclusions. When a test fails, identify the records or values responsible whenever the question asks for exceptions. Checks should **report** issues rather than terminate the notebook. Do not assume that the data is correct just because the JSON files load successfully.

### Check 1 — Verify the stock universe

Write code to:
- Report the number of tickers and the number of unique tickers.
- Determine whether there are **exactly 100 unique tickers**.
- Print each duplicated ticker, if any.

**In your code:** Add a Python comment explaining why checking only `len(tickers) == 100` is insufficient.

In [ ]:
# Check 1 — write your code here


### Check 2 — Corporate-action ticker coverage

Write code to determine whether the keys of `corporate_actions` contain exactly the tickers in the stock universe. Print a Boolean result, any **missing** tickers, and any **extra** tickers.

**Explain:** What does a missing ticker mean for the transformation?

In [ ]:
# Check 2 — write your code here


### Check 3 — EOD ticker coverage

Compare the keys of `eod_prices` with the stock universe. Print a Boolean result and any missing or extra tickers. Do not reuse the answer from Check 2; investigate this source separately.

In [ ]:
# Check 3 — write your code here


### Check 4 — EOD observation counts

For each ticker in the EOD dictionary, count its records. Report:
- How many tickers have **exactly 100** observations.
- How many do not.
- The ticker and actual count for every exception.

**Explain:** Why does the transformation depend on consistent observation counts?

In [ ]:
# Check 4 — write your code here


### Check 5 — Date consistency, ordering, and uniqueness

Write code for **all four** tasks:

**5A.** Choose the first ticker in `tickers` and create a reference list of dates from its EOD records. Use the first 10 characters of each record's `"Date"` field.

**5B.** Compare the complete date sequence for every ticker with that reference sequence. Count matches and nonmatches. For each nonmatch, report the ticker and either its first differing index/date or a length mismatch.

**5C.** Determine whether the reference dates are in ascending order.

**5D.** Determine whether the reference dates are unique. Print duplicated dates if any.

**Explain:** Why is matching observation counts not enough to guarantee alignment?

In [ ]:
# Check 5A — reference date sequence


In [ ]:
# Check 5B — compare the sequences across tickers


In [ ]:
# Check 5C — chronological order


In [ ]:
# Check 5D — duplicate dates


### Check 6 — Closing-price quality

Inspect the `"Close"` field of **every EOD record** and separately count:
- Missing `"Close"` keys
- Values equal to `None`
- Nonnumeric values (consider Python `bool` values nonnumeric for this purpose)
- Zero prices
- Negative prices

Print the ticker, date, and offending value for each problem. Print the five final counts.

**Explain:** Why might a numerically valid value still be unsuitable for a stock price?

In [ ]:
# Check 6 — write your code here


### Check 7 — Corporate-action exploration

Inspect each ticker's corporate-action records. The supplied source uses `"Dividends"`, `"Stock Splits"`, and `"Date"` fields.

Write code to report:
- The distribution of dividend-event counts by ticker, using a six-element list for **0, 1, 2, 3, 4, and 5 or more** events.
- The corresponding distribution of split-event counts.
- The number of tickers that have **both** at least one dividend event and at least one split event.
- The number of distinct corporate-action dates.
- Every date with more than one corporate-action record, together with its record count.
- Whether each six-element distribution sums to 100.

Treat a nonzero `"Dividends"` as a dividend event and a nonzero `"Stock Splits"` as a split event.

**Explain:** What do the six positions in your distribution lists represent? Why is the last position different from the first five?

In [ ]:
# Check 7 — write your code here


### Check 8 — Corporate-action date alignment

Compare the set of corporate-action dates with the set of EOD dates from Check 5. Report whether **every corporate-action date** occurs in the EOD sequence; print any missing dates.

**Explain:** Why would a corporate action on a date not represented in the EOD source complicate construction of `market_data`?

In [ ]:
# Check 8 — write your code here


## Part B — Construct `market_data`

**Objective:** Reorganize end-of-day (EOD) data by trading day and combine it with the corresponding corporate actions. Before writing the transformation, study the new dictionary's shape and how to access its values.

### B1. The fundamental change: ticker-first to day-first

**Before:** `eod_prices` is organized by **ticker**, then observation/day.  
**After:** `market_data` is organized by **trading day**, then ticker.

```text
BEFORE: eod_prices                  AFTER: market_data
Organized by TICKER                 Organized by TRADING DAY

eod_prices                          market_data
|                                   |
+-- TSM                              +-- Day 0
|   +-- Day 0: Price                |   +-- date: 2025-08-15
|   +-- Day 1: Price                |   +-- TSM: Price, Div, Split
|   +-- Day 2: Price                |   +-- AAPL: Price, Div, Split
|   +-- ...                         |   +-- ... 98 more tickers
|   +-- Day 99: Price               |
|                                   +-- Day 1
+-- AAPL                             |   +-- date: 2025-08-18
|   +-- Day 0: Price                |   +-- TSM: Price, Div, Split
|   +-- Day 1: Price                |   +-- AAPL: Price, Div, Split
|   +-- Day 2: Price                |   +-- ... 98 more tickers
|   +-- ...                         |
|   +-- Day 99: Price               +-- ...
|                                   |
+-- ... 98 more tickers             +-- Day 99
```

The original access pattern is:

```python
eod_prices[ticker][day]["Close"]
```

Here `day` refers to a **position in a ticker's list of daily observations**, assuming the records are aligned and ordered (as investigated in Part A).

The new access pattern is:

```python
market_data[day][ticker]["price"]
market_data[day][ticker]["div"]
market_data[day][ticker]["split_in"]
market_data[day][ticker]["split_out"]
```

**What changes?** We start with 100 tickers, each containing 100 daily observations, and reorganize them into 100 trading days, each containing 100 ticker records **plus a date field**. We also merge in dividend and stock-split information from `corporate_actions`; this is more than simply reversing two indexes.

**Why?** PEMS advances one trading day at a time. With the day-first structure, the system can retrieve all information for a given trading day from one daily dictionary.

### B2. The target nested-dictionary structure

This is a **small illustrative excerpt**. The finished `market_data` must have all 100 tickers on each of days `0` through `99`.

```python
market_data = {
    0: {
        "date": "2025-08-15",
        "TSM": {"price": 238.88, "div": 0, "split_in": 1, "split_out": 1},
        "BRK.B": {"price": 477.20, "div": 0, "split_in": 1, "split_out": 1},
        "LLY": {"price": 701.23, "div": 1.5, "split_in": 1, "split_out": 1},
        # ... 97 additional tickers
    },
    1: {
        "date": "2025-08-18",
        "TSM": {"price": 241.41, "div": 0, "split_in": 1, "split_out": 1},
        "BRK.B": {"price": 478.52, "div": 0, "split_in": 1, "split_out": 1},
        "LLY": {"price": 698.05, "div": 0, "split_in": 1, "split_out": 1},
        # ... 97 additional tickers
    },
    # ... through day 99
}
```

The example is provided to **explain the intended shape**, not as data to copy into the solution. Retrieve dates, prices, dividends, and split details from the files.

**Three levels of dictionary access:**

```text
market_data
  +-- 0  (trading day)
  |   +-- "date" --> "2025-08-15"
  |   +-- "TSM"  --> {"price": ..., "div": ..., "split_in": ..., "split_out": ...}
  |   +-- "LLY"  --> {"price": ..., "div": ..., "split_in": ..., "split_out": ...}
  |   +-- ... 98 more tickers
  +-- 1  (next trading day)
  +-- ... through day 99
```

- **Level 1:** `market_data[day]` returns the **daily-record dictionary**.
- **Level 2:** `market_data[day][ticker]` returns the **ticker-record dictionary** for that day.
- **Level 3:** `market_data[day][ticker]["price"]` returns a **single numeric value**. Substitute `"div"`, `"split_in"`, or `"split_out"` to retrieve those fields.

Each daily-record dictionary also has a `"date"` key: `market_data[day]["date"]`. Thus, it contains **101 keys**, not 100: one date and 100 tickers. Unlike the ticker values, the date value is a string.

### B3. Reading nested dictionaries one step at a time

For the example above, let `day = 0` and `ticker = "LLY"`.

| Expression | What it returns | Type |
|---|---|---|
| `market_data` | Dictionary of all trading days | `dict` |
| `market_data[day]` | Date plus all 100 ticker records for day 0 | `dict` |
| `market_data[day]["date"]` | `"2025-08-15"` | `str` |
| `market_data[day][ticker]` | `{"price": 701.23, "div": 1.5, "split_in": 1, "split_out": 1}` | `dict` |
| `market_data[day][ticker]["price"]` | `701.23` | `float` |
| `market_data[day][ticker]["div"]` | `1.5` | `float` |

Python accesses these dictionaries **one key at a time**. The following expressions are equivalent:

```python
price = market_data[day][ticker]["price"]
```

```python
daily_record = market_data[day]
ticker_record = daily_record[ticker]
price = ticker_record["price"]
```

**Important:** Do not treat every key inside `market_data[day]` as a ticker. The key `"date"` maps to a string, not a ticker-record dictionary. Code that tries `market_data[day]["date"]["price"]` will fail.

### B4. Plan the transformation before coding

Decide how to solve these questions. Explain your choices **in Python comments in the implementation cell**, not in new Markdown answer cells.

1. Should the outer loop move through tickers or trading days? Why?
2. When will you create each new daily-record dictionary, and where will you get its date?
3. How will you retrieve the right EOD closing price for a ticker and trading day?
4. How will you match corporate actions to the **same ticker and date**?
5. What values will you use when there is no dividend or split?
6. How will you avoid treating `"date"` as a ticker?

Your code should build the dictionary from the source data; it should not hard-code the illustrative records.

### B5. Implementation requirements

Using the loaded `tickers`, `corporate_actions`, and `eod_prices`, write Python code to create a dictionary named `market_data`.

**Required structure** (illustrative values only):

```python
{
    0: {
        "date": "2025-08-15",
        "TSM": {"price": 238.88, "div": 0, "split_in": 1, "split_out": 1},
        "BRK.B": {"price": 477.20, "div": 0, "split_in": 1, "split_out": 1},
        # ... all 100 tickers
    },
    1: {
        "date": "2025-08-18",
        # ... all 100 tickers
    },
    # ... through trading day 99
}
```

Requirements:
1. Use **integer** trading-day keys from `0` through `99`, in chronological order.
2. Each day must contain a `"date"` string in `YYYY-MM-DD` format and an entry for **every ticker**.
3. Each ticker's `"price"` is that day's EOD `"Close"` value.
4. Attach the dividend and split information for that **ticker and date**. Use `div = 0`, `split_in = 1`, `split_out = 1` when no applicable action exists.
5. Examine the corporate-action records before implementing split conversion. Use Python comments in your transformation cell to explain how you interpret `"Stock Splits"` and produce `split_in` and `split_out`, including any limitations of the source representation. Do not invent action values.
6. Build the dictionary from the files; do **not** type in the example prices or dates.

Write the transformation in the following cell. You may create small helper functions if useful.

In [ ]:
# Part B — implement the transformation
market_data = {}

# Your code here


### Test your constructed dictionary

Write your **own** tests, not just a visual inspection. At minimum, show that:
- The day keys are `0` through `99`.
- The dates correspond to the EOD reference sequence.
- Each day contains exactly 100 ticker entries plus the `"date"` field.
- Each ticker record has exactly the four required fields.
- Selected prices match the source EOD records.
- Corporate-action values match the source on dates with actions, and defaults appear on dates without actions.

Print a small sample of your result (for example, five tickers on days 0 and 1) rather than all 10,000 ticker-day records.

In [ ]:
# Write your market_data tests here
